# 01 Data loading

Loads the saved train/validation/test splits from `data/splits/` (made once by `00_data_splits.ipynb`: 80/10/10, stratified by family) with `src/data/dataset.py`, which checks every file against `manifest.json`, and computes the sample weights. Every model loads the data this way, so all models see the same rows.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
from src.data.dataset import load_splits

train, val, test = load_splits()
train.head()

,domain,label,family,weight
0,004a512fa73707a0a20da6ff6d64f1ab.co.cc,dga,bamital,1.662579
1,004a512fa73707a0a20da6ff6d64f1ab.info,dga,bamital,1.662579
2,005c829b446f26a4787b0e5ba432e57d.org,dga,bamital,1.662579
3,0064c05a8f474b4fc1ff70b3bdfe6011.org,dga,bamital,1.662579
4,00b67ad0aa338175ee74c01fdd9cde99.org,dga,bamital,1.662579


## Split sizes

In [3]:
pd.DataFrame({name: df["label"].value_counts() for name, df in
              [("train", train), ("val", val), ("test", test)]}).T

label,dga,legit
train,799737,799731
val,99967,99966
test,99967,99967


## Families

`legit`, `other_dga` (the 59 families with fewer than 300 domains) and 102 DGA families, each with 300 to 29,120 domains.

In [4]:
fam = train.loc[train["label"] == "dga", "family"].value_counts()
print(f"{fam.size} DGA families in train, {fam.min()} to {fam.max()} domains each")
fam.describe()

103 DGA families in train, 240 to 23296 domains each


count      103.000000
mean      7764.436893
std       9061.620487
min        240.000000
25%        742.500000
50%       3072.000000
75%      14216.000000
max      23296.000000
Name: count, dtype: float64

## Sample weights

Legitimate and DGA weigh half each; within DGA a domain's weight is proportional to 1/sqrt(family size), so small families count more without dominating.

In [5]:
w = train.groupby("label")["weight"].agg(["sum", "min", "max"])
dga_w = train.loc[train["label"] == "dga", "weight"]
print(f"DGA weight max/min: {dga_w.max() / dga_w.min():.1f}")
w

DGA weight max/min: 9.9


,sum,min,max
label,,,
dga,799734.0,0.702990,6.926028
legit,799734.0,1.000004,1.000004
